진행 순서
1. 나만의 데이터 셋 준비하기
2. torchvision.datasets.ImageFolder으로 불러오기
3. transforms 적용해 저장하기 origin_data -> train_data

현재 데이터셋 : connect에 있는 의자
- 문제점 : 너무 커서 크기를 줄여야 함.

In [25]:
import torchvision
from torchvision import transforms

from torch.utils.data import DataLoader

from matplotlib.pyplot import imshow
%matplotlib inline

In [26]:
trans = transforms.Compose([
    transforms.Resize((64, 128))
])
train_data = torchvision.datasets.ImageFolder(root='custom_data/origin_data', transform=None)

In [27]:
for num, value in enumerate(train_data):
    data, label = value
    print(num, data, label)

    if (label==0):
        data.save('custom_data/train_data/gray/%d_%d.jpeg'%(num, label))
    else:
        data.save('custom_data/train_data/red/%d_%d.jpeg'%(num, label))

0 <PIL.Image.Image image mode=RGB size=512x256 at 0x1DAC0001FF0> 0
1 <PIL.Image.Image image mode=RGB size=512x256 at 0x1DAC0001300> 0
2 <PIL.Image.Image image mode=RGB size=512x256 at 0x1DAC0001FF0> 0
3 <PIL.Image.Image image mode=RGB size=512x256 at 0x1DAC0001300> 0
4 <PIL.Image.Image image mode=RGB size=512x256 at 0x1DAC0001FF0> 0
5 <PIL.Image.Image image mode=RGB size=512x256 at 0x1DAC0001300> 0
6 <PIL.Image.Image image mode=RGB size=512x256 at 0x1DAC0001FF0> 0
7 <PIL.Image.Image image mode=RGB size=512x256 at 0x1DAC0001300> 0
8 <PIL.Image.Image image mode=RGB size=512x256 at 0x1DAC0001FF0> 0
9 <PIL.Image.Image image mode=RGB size=512x256 at 0x1DAC0001300> 0
10 <PIL.Image.Image image mode=RGB size=512x256 at 0x1DAC0001FF0> 0
11 <PIL.Image.Image image mode=RGB size=512x256 at 0x1DAC0001300> 0
12 <PIL.Image.Image image mode=RGB size=512x256 at 0x1DAC0001FF0> 0
13 <PIL.Image.Image image mode=RGB size=512x256 at 0x1DAC0001300> 0
14 <PIL.Image.Image image mode=RGB size=512x256 at 0x1DAC0

# Neural Network 만들기 

할 일
- 사진을 이용해 dataset 만드는 법
- 배운 내용 이용해 학습하기
- 모델 저장하고 다시 불러오기 

In [28]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import torch.optim as optim
from torch.utils.data import DataLoader

import torchvision
import torchvision.transforms as transforms

In [29]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

torch.manual_seed(777)
if device =='cuda':
    torch.cuda.manual_seed_all(777)

In [36]:
trans = transforms.Compose([
    transforms.Resize((64, 128)),
    transforms.ToTensor()
])

train_data = torchvision.datasets.ImageFolder(root='./custom_data/train_data', transform=trans)


In [31]:
data_loader = DataLoader(dataset=train_data, batch_size=8, shuffle=True, num_workers=2)

In [32]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(3, 6, 5),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.layer2 = nn.Sequential(
            nn.Conv2d(6, 16, 5),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.layer3 = nn.Sequential(
            nn.Linear(6032, 120),
            nn.ReLU(),
            nn.Linear(120, 2)
        )

    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = out.view(out.shape[0], -1)
        out = self.layer3(out)
        return out

In [37]:
# testing
net = CNN().to(device)
test_input = (torch.Tensor(3, 3, 64, 128)).to(device)
test_out = net(test_input)

In [38]:
optimizer = optim.Adam(net.parameters(), lr=0.00005)
loss_func = nn.CrossEntropyLoss().to(device)

In [39]:
total_batch = len(data_loader)

epochs = 7
for epoch in range(epochs):
    avg_cost = 0.0
    for num, data in enumerate(data_loader):
        imgs, labels = data
        imgs = imgs.to(device)
        optimizer.zero_grad()
        out = net(imgs)
        loss = loss_func(out, labels)
        loss.backward()
        optimizer.step()

        avg_cost += loss / total_batch
    print('[Epoch:{}] cost = {}'.format(epoch+1, avg_cost))
print('Learning Finished!')   

RuntimeError: mat1 and mat2 shapes cannot be multiplied (8x122000 and 6032x120)

In [ ]:
torch.save(net.state_dict(), "/model/model.pth")

In [ ]:
new_net = CNN().to(device)
new_net.load_state_dict(torch.load('./model/model.pth'))

In [ ]:

print(net.layer1[0])
print(new_net.layer1[0])

print(net.layer1[0].weight[0][0][0])
print(new_net.layer1[0].weight[0][0][0])

net.layer1[0].weight[0] == new_net.layer1[0].weight[0]

In [ ]:
trans=torchvision.transforms.Compose([
    transforms.Resize((64,128)),
    transforms.ToTensor()
])
test_data = torchvision.datasets.ImageFolder(root='./custom_data/test_data', transform=trans)

In [ ]:
test_set = DataLoader(dataset = test_data, batch_size = len(test_data))

In [ ]:
with torch.no_grad():
    for num, data in enumerate(test_set):
        imgs, label = data
        imgs = imgs.to(device)
        label = label.to(device)
        
        prediction = net(imgs)
        
        correct_prediction = torch.argmax(prediction, 1) == label
        
        accuracy = correct_prediction.float().mean()
        print('Accuracy:', accuracy.item())